# VAEs and GANs: Two Ways to Sample What You Can't Write Down

The full code of [VAEs and GANs: Two Ways to Sample What You Can't Write Down](https://residual-stream.lvtan71.workers.dev/posts/vaes-and-gans/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/vaes-and-gans) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vaes-and-gans.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 12 min end to end on the RTX 3090 machine that saved the outputs below.
With three seeds, which seed fails to train depends on the machine.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 12 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "vaes-and-gans"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/vaes-and-gans


## `elbo_check.py`

A 2-D linear-Gaussian latent model solved exactly: log p(x), the true posterior, and for four Gaussian q(z\|x) the sampled and exact reconstruction term, the KL to the prior, the ELBO, and its gap to log p(x) next to the KL to the true posterior.

In [2]:
!{sys.executable} elbo_check.py

log p(x), exact: -2.3429
true posterior: mean (0.365, 0.618), std (0.442, 0.479), correlation -0.210

q(z|x)                               recon (sampled, exact) KL to prior     ELBO     gap KL to post


best diagonal Gaussian                  -1.3081    -1.3066       1.0590  -2.3655  0.0226     0.0226
posterior's mean and marginal stds      -1.3417    -1.3435       1.0231  -2.3666  0.0237     0.0237


right mean, stds 3x too wide            -8.0050    -8.0153       0.5245  -8.5398  6.1969     6.1969
the prior, N(0, I)                      -6.1847    -6.1916       0.0000  -6.1916  3.8487     3.8487

KL(N(mu, sigma^2) || N(0, I)) for mu = (1, -0.5), sigma = (0.5, 2): 1.7500 nats


The post printed:

```text
log p(x), exact: -2.3429
true posterior: mean (0.365, 0.618), std (0.442, 0.479), correlation -0.210

q(z|x)                               recon (sampled, exact) KL to prior     ELBO     gap KL to post
best diagonal Gaussian                  -1.3081    -1.3066       1.0590  -2.3655  0.0226     0.0226
posterior's mean and marginal stds      -1.3417    -1.3435       1.0231  -2.3666  0.0237     0.0237
right mean, stds 3x too wide            -8.0050    -8.0153       0.5245  -8.5398  6.1969     6.1969
the prior, N(0, I)                      -6.1847    -6.1916       0.0000  -6.1916  3.8487     3.8487

KL(N(mu, sigma^2) || N(0, I)) for mu = (1, -0.5), sigma = (0.5, 2): 1.7500 nats
```

## `gen2d.py`

The GAN's value at the optimal discriminator checked against 2 JS − log 4; then a VAE, a non-saturating GAN and WGAN-GP trained on 8 Gaussian modes on a ring, 6,000 generator steps, 3 seeds each, scored by modes covered and share of samples near a mode.

In [3]:
!{sys.executable} gen2d.py

optimal-discriminator value -0.9440; 2 JS - log 4 = -0.9440

model                    modes covered (of 8), per seed   share of samples near a mode


VAE                                 8  8  8            52%  48%  53%


GAN (non-saturating)                8  8  8            89%  53%  88%


WGAN-GP                             0  8  8            2%  69%  66%


The post printed:

```text
optimal-discriminator value -0.9440; 2 JS - log 4 = -0.9440

model                    modes covered (of 8), per seed   share of samples near a mode
VAE                                 8  8  8            52%  48%  51%
GAN (non-saturating)                8  7  8            89%  82%  88%
WGAN-GP                             8  8  1            72%  65%  2%
```

## `vae_codes.py`

Retrains `gen2d.py`'s VAE for the same 3 seeds and scores decoded reconstructions, codes the encoder could give, and codes from the prior. Loads `gen2d.py`.

In [4]:
!{sys.executable} vae_codes.py

seed  reconstructions  z from the encoders  z from the prior


   0             100%                 100%               52%


   1             100%                  99%               50%


   2             100%                 100%               51%


The post printed:

```text
seed  reconstructions  z from the encoders  z from the prior
   0             100%                 100%               52%
   1             100%                  99%               50%
   2             100%                 100%               48%
```

## `wgan_long.py`

WGAN-GP seed 2 run to 12,000 generator steps and scored every 1,500. Loads `gen2d.py`.

In [5]:
!{sys.executable} wgan_long.py

step   1500   modes 0   near a mode  0.9%   sample std (1.56, 0.84)


step   3000   modes 8   near a mode 35.2%   sample std (1.38, 1.34)


step   4500   modes 8   near a mode 55.6%   sample std (1.38, 1.37)


step   6000   modes 8   near a mode 67.0%   sample std (1.40, 1.38)


step   7500   modes 8   near a mode 76.2%   sample std (1.41, 1.41)


step   9000   modes 8   near a mode 78.9%   sample std (1.38, 1.42)


step  10500   modes 8   near a mode 85.0%   sample std (1.41, 1.40)


step  12000   modes 8   near a mode 85.7%   sample std (1.40, 1.42)


The post printed:

```text
step   1500   modes 0   near a mode  0.1%   sample std (0.69, 0.43)
step   3000   modes 0   near a mode  0.8%   sample std (3.05, 2.17)
step   4500   modes 0   near a mode  0.0%   sample std (0.16, 0.24)
step   6000   modes 0   near a mode  0.0%   sample std (0.27, 0.63)
step   7500   modes 0   near a mode  1.0%   sample std (0.99, 2.04)
step   9000   modes 0   near a mode  0.2%   sample std (1.05, 1.03)
step  10500   modes 0   near a mode  0.4%   sample std (0.44, 0.83)
step  12000   modes 0   near a mode  1.4%   sample std (0.89, 1.00)
```